System Checking, including python veresion, docker, docker compose and uv checking

In [4]:
# Find Project Root
import sys
from pathlib import Path

current_dir = Path.cwd()
print(f"current_dir= {current_dir}")

if current_dir.name == "test":
    project_root = current_dir.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = None

if project_root and (project_root / "compose.yml").exists():
    print(f"✓ Project root: {project_root}")
else:
    print("✗ Missing compose.yml - check directory")
    exit()

current_dir= /Users/xieqiqi/Learning/LLM/my_first_Rag_project/test
✓ Project root: /Users/xieqiqi/Learning/LLM/my_first_Rag_project


In [5]:
# Check Python version
import sys
from pathlib import Path 

python_version = sys.version_info
print(f"Python version: {python_version.major}.{python_version.minor}.{python_version.micro}")
print(f"Environment: {sys.executable}")

if python_version >= (3, 12):
    print("You are using Python 3.12 or higher.")
else:
    print("You are using an older version of Python.")

Python version: 3.12.11
Environment: /Users/xieqiqi/Learning/LLM/my_first_Rag_project/.venv/bin/python3
You are using Python 3.12 or higher.


In [6]:
# Docker check
import subprocess

try:
    result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Docker: {result.stdout}")
    else:
        print("Docker: Not working")
        exit()
except:
    print("Docker: Not found")
    exit()


Docker: Docker version 20.10.22, build 267d2e5



In [7]:
# Check docker compose
try:
    result = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Docker Compose: {result.stdout.split()[3]}")
    else:
        print("Docker Compose: Not working")
        exit()
except:
    print("Docker Compose: Not found")
    exit()

Docker Compose: v2.20.2-desktop.1


In [8]:
# Check uv version
import subprocess
try:
    result = subprocess.run(["uv", "--version"  ], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"UV: {result.stdout.strip()}")
        print("\n✓ All required software ready!")
    else:
        print("UV: Not working")
        exit()
except:
    print("UV: Not found")
    exit()

UV: uv 0.8.19 (fc7c2f8b5 2025-09-19)

✓ All required software ready!


Start service

```bash
make start 
```
or 

```bash
docker compose up -d
```



In [1]:
# Check Docker running
try:
    result = subprocess.run(["docker", "info"], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Docker is running")
    else:
        print("Docker not running - start Docker Desktop")
        exit()
except:
    print("Docker daemon not accessible")
    exit()


Docker daemon not accessible


In [ ]:
# Check container status
import json
try:
    result = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"],
        cwd=str(project_root),
        capture_output=True,
        text=True,
        timeout=10
    )
    if result.returncode == 0 and result.stdout.strip():
        print("Current containers:")
        containers = json.loads(result.stdout)
        for container in containers:
            service = container.get("Service", "Unknown")
            status = container.get("Status", "Unknown")
            print(f"{service}: {status}")
    else:
        print("No containers running")
except:
    print("Could not check containers")

Service Health Check

In [ ]:
EXPECTED_SERVICES = {
    'api': 'FastAPI REST API server',
    'postgres': 'PostgreSQL database',
    'opensearch': 'OpenSearch search engine', 
    'opensearch-dashboards': 'OpenSearch web dashboard',
    'ollama': 'Local LLM inference server',
    'airflow': 'Workflow automation (optional - may be off)'
}

try:
    result = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"],
        cwd=str(project_root),
        capture_output=True,
        text=True,
        timeout=15
    )
    
    if result.returncode == 0:
        print("SERVICE STATUS")
        print("=" * 70)
        print(f"{'Service':<20} {'State':<15} {'Status':<15} {'Notes'}")
        print("-" * 70)
    else:
        print("Could not get service status")
        exit()
        
except Exception as e:
    print(f"Error checking services: {e}")
    exit()

# Parse Service Status
found_services = set()
service_states = {}

if result.stdout.strip():
    try:
        containers = json.loads(result.stdout)
        for container in containers:
            service = container.get('Service', 'unknown')
            state = container.get('State', 'unknown')
            health = container.get('Health', 'no check')
            found_services.add(service)
            service_states[service] = {'state': state, 'health': health} 
            if state == 'running' and health in ['healthy', 'no check']:
                indicator = "✓"
                notes = "Ready"
            elif state == 'running' and health == 'unhealthy':
                indicator = "⚠"
                notes = "Starting up..."
            elif state == 'exited':
                indicator = "✗"
                notes = "Failed to start"
            else:
                indicator = "?"
                notes = f"Status: {state}"
            
            print(f"{indicator} {service:<18} {state:<14} {health:<14} {notes}")
    except  json.JSONDecodeError:
        print("Could not parse service status")
        pass


PostgreSQL - Database Storage


In [ ]:
# Check PostgreSQL connection
import socket
def test_postgresql_connection():
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(5)
        result = sock.connect_ex(('localhost', 5432))
        sock.close()
        if result == 0:
            print("✓ PostgreSQL is accepting connections on port 5432!")
            return True
        else:
            print("✗ PostgreSQL is not accepting connections on port 5432.")
            return False
    except Exception as e:
        print(f"✗ Could not connect to PostgreSQL: {e}")
        return False 

postgresql_connection = test_postgresql_connection()

if postgresql_connection:
    print(f"\n Database connection details:")
    print("• Host: localhost")
    print("• Port: 5432") 
    print("• Database: rag_db")
    print("• Username: rag_user")
    print("• Password: rag_password")
    
    print("\n  Recommended GUI Tools:")
    print("• DBeaver (Free): https://dbeaver.io/download/")
    print("• pgAdmin: https://www.pgadmin.org/download/")

In [ ]:
# Test PostgreSQL connection
import psycopg2
try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        user="rag_user",
        password="rag_password",
        dbname="rag_db"
    )
    print("PostgreSQL connection successful")
except Exception as e:
    print(f"PostgreSQL connection failed: {e}")


In [ ]:
cursor = conn.cursor()
cursor.execute("""select table_name from information_schema.tables where table_schema='public' order by table_name""")

all_tables = cursor.fetchall()

app_tables = []
airflow_tables = []

for (table_name,) in all_tables:
    if table_name in ['papers', 'users', 'embeddings']:
        app_tables.append(table_name)
    else:
        airflow_tables.append(table_name)
    
print(f"Found {len(all_tables)} All Tables:")
print(f"Application tables: {len(app_tables)}")
print(f"Airflow tables: {len(airflow_tables)}")


for table in app_tables:
    print(f"  • {table}")

if not app_tables:
    print("  No application tables yet (expected in Week 1)")
    
cursor.close()
conn.close()

# FastAPI - API service checking

API documentation: http://localhost:8000/docs

In [ ]:
# Test FastAPI health check
import requests

try:
    response = requests.get("http://localhost:8000/api/v1/health", timeout=5)
    if response.status_code == 200:
        data = response.json()
        print("✓ FastAPI is responding")
        print(f"Status: {data.get('status', 'unknown')}")
    else:
        print(f"⚠ API returned status: {response.status_code}")
except requests.exceptions.ConnectionError as e:
    print(f"⚠ API not responding: {e}")
except Exception as e:
    print(f"⚠ API test error: {e}") 


# Airflow Checking
Web dashboard: http://localhost:8080


In [ ]:
import json
from pathlib import Path

password_file = project_root / "airflow" / "simple_auth_manager_passwords.json.generated"

try:
    if password_file.exists():
        with password_file.open("r") as f:
            password = json.load(f)
            print(password)
    else:
        password = "admin"
    print(f"Password is {password}")
except Exception as e:
    print(f"Could not read password: {e}")
    password = None


In [ ]:
# Test Airflow health

try:
    response = requests.get("http://localhost:8080/health")
    if response.status_code == 200:
        print("Airflow is healthy")
        if password:
            print(f"\nAirflow Login:")
            print(f"URL: http://localhost:8080")
            print(f"Username: admin")
            print(f"Password: {password}")
    else:
        print(f"Airflow health check failed with status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print(f"✗ Airflow not responding - wait 2-3 minutes")
except Exception as e:
    print(f"✗ Airflow health check failed: {str(e)}")


In [ ]:
# Ollama
API endpoint: http://ollama:11434

In [16]:
# Check Ollama Service Status
import requests
import json

ollama_url = "http://localhost:11434/api/tags"
try:
    response = requests.get(ollama_url)
    if response.status_code == 200:
        models_data = response.json()
        models = models_data.get("models", [])

        print("Ollama service is running")
        print(f"Available models: {len(models)}")

        if models:
            print("\nInstalled Models:")
            for model in models:
                name = model.get('name', 'unknown')
                size = model.get('size', 0)
                size_gb = round(size / (1024**3), 1)
                print(f"  • {name} ({size_gb} GB)")
        else:
            print("\n  No models installed yet")
            print("   This is normal - models are large files (3-7 GB each)")
            print("   In Week 4, we'll install a model like llama3.2")
        print("\n  Try This Later (Week 4):")
        print("1. docker exec -it rag-ollama ollama pull llama3.2")
        print("2. docker exec -it rag-ollama ollama list")
        print("3. docker exec -it rag-ollama ollama run llama3.2")
        
    else:
        print(f"Ollama returned status: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("Ollama service is not responding yet")
    print("Ollama service might still be starting")
except requests.exceptions.Timeout:
    print(f"Ollama request timed out")
    print("Service might still be initializing")
except Exception as e:
    print(f"✗ Unexpected error testing Ollama: {e}")
    print("Try again in a few minutes")

Ollama service is running
Available models: 0

  No models installed yet
   This is normal - models are large files (3-7 GB each)
   In Week 4, we'll install a model like llama3.2

  Try This Later (Week 4):
1. docker exec -it rag-ollama ollama pull llama3.2
2. docker exec -it rag-ollama ollama list
3. docker exec -it rag-ollama ollama run llama3.2


In [17]:
# Check Ollama version and health
import requests
import json

ollama_version_url = "http://localhost:11434/api/version"

try:
    response = requests.get(ollama_version_url, timeout=5)
    if response.status_code == 200:
        version_data = response.json()
        version = version_data.get('version', 'unknown')
        
        print("✓ Ollama API is healthy!")
        print(f"Version: {version}")
        
        print("\n  What is Ollama?")
        print("• Runs AI models completely on your local machine")
        print("• No data sent to external services (privacy-first)")
        print("• No API fees or rate limits")
        print("• Supports models like Llama, Mistral, Phi, etc.")
        
        print("\n  Coming in Week 4:")
        print("• Install and run a local language model")
        print("• Generate answers to research questions")
        print("• Summarize academic papers")
        print("• All processing stays on your computer!")
    else:
         print(f"⚠ Ollama version check returned: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("✗ Could not check Ollama version")
    print("Service might still be starting up")
    
except requests.exceptions.Timeout:
    print("✗ Ollama request timed out")
    print("Service might still be initializing")
    
except Exception as e:
    print(f"✗ Unexpected error checking version: {e}")
    print("Try again in a few minutes")

✓ Ollama API is healthy!
Version: 0.13.5

  What is Ollama?
• Runs AI models completely on your local machine
• No data sent to external services (privacy-first)
• No API fees or rate limits
• Supports models like Llama, Mistral, Phi, etc.

  Coming in Week 4:
• Install and run a local language model
• Generate answers to research questions
• Summarize academic papers
• All processing stays on your computer!
